In [31]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [32]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from imports import *
from config import dir_config, main_config
from src.glm_hmm.cv_utils import bernoulli_null_loglik, pooled_bits_per_trial, select_best_n_states

In [33]:
processed_dir = Path(dir_config.data.processed)
glm_hmm_dir = processed_dir / "glm_hmm"

MODEL_PATHS = {f.stem: f for f in glm_hmm_dir.iterdir() if f.is_dir()}

In [34]:
def session_bits(bundle):
    """Session-wise CV: bits/trial computed per session then averaged (mean, SEM across sessions)."""
    sw = bundle["session_wise"]
    states = np.array(list(sw["models"][0].keys()))
    n_trials, null = [], []
    for df in bundle["data"].values():
        valid = ~df["invalid_idx"].values
        y = df["choices"].values[valid]
        p = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        n_trials.append(int(valid.sum()))
        null.append((y * np.log(p) + (1 - y) * np.log(1 - p)).sum())
    n_trials, null = np.array(n_trials), np.array(null)
    bits = (sw["test_ll"].sum(2) - null[:, None]) / (n_trials[:, None] * np.log(2))
    return states, bits.mean(0), bits.std(0) / np.sqrt(bits.shape[0])

def session_train_bits(bundle):
    """Session-wise CV training bits/trial, averaged over sessions and folds.

    Train folds overlap (each leaves out one fold), so they cannot be summed into one pass
    like the test folds. We score each (session, state, fold) and average; the per-fold train
    size and null are approximated as (k-1)/k of the session's valid trials (k-fold split).
    """
    sw = bundle["session_wise"]
    states = np.array(list(sw["models"][0].keys()))
    k_folds = sw["train_ll"].shape[2]
    frac = (k_folds - 1) / k_folds
    n_trials, null = [], []
    for df in bundle["data"].values():
        valid = ~df["invalid_idx"].values
        y = df["choices"].values[valid]
        p = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        n_trials.append(int(valid.sum()))
        null.append((y * np.log(p) + (1 - y) * np.log(1 - p)).sum())
    n_trials = np.array(n_trials)[:, None, None]
    null = np.array(null)[:, None, None]
    bits_fold = (sw["train_ll"] - null * frac) / (n_trials * frac * np.log(2))  # (sessions, states, folds)
    n = np.sum(~np.isnan(bits_fold), axis=(0, 2))
    return states, np.nanmean(bits_fold, axis=(0, 2)), np.nanstd(bits_fold, axis=(0, 2)) / np.sqrt(n)

def pooled_bits(bundle):
    """Pooled CV: (states, mean_bits, sem_bits) over one held-out pass, vs the Bernoulli null."""
    gpc = bundle["group_pooled_cv"]
    null_ll, n_valid = bernoulli_null_loglik(bundle["data"])
    mean_bits, sem_bits = pooled_bits_per_trial(gpc["test_ll"], gpc["n_test_trials"], null_ll, n_valid)
    return np.asarray(gpc["state_range"]), mean_bits, sem_bits

def pooled_train_bits(bundle):
    """Pooled CV training bits/trial per state (mean, SEM across folds), vs the Bernoulli null.

    Train folds overlap, so they are scored per fold (each normalized by its own train-trial
    count and a proportional null) and averaged, rather than summed into one pass.
    """
    gpc = bundle["group_pooled_cv"]
    null_ll, n_valid = bernoulli_null_loglik(bundle["data"])
    train_ll, n_train = gpc["train_ll"], gpc["n_train_trials"]
    bits_fold = (train_ll - null_ll * (n_train / n_valid)) / (n_train * np.log(2))  # (states, folds)
    return np.asarray(gpc["state_range"]), bits_fold.mean(1), bits_fold.std(1) / np.sqrt(bits_fold.shape[1])

def parsimonious_k(states, mean_bits, sem_bits):
    """Smallest K within 1 SEM (or tol) of the peak; mirrors cv_utils.select_best_n_states."""
    best_idx = int(np.argmax(mean_bits))
    cutoff = mean_bits[best_idx] - sem_bits[best_idx]
    chosen_idx = next(i for i in range(best_idx + 1) if mean_bits[i] >= cutoff)
    return int(states[chosen_idx]), int(states[best_idx])

def plot_test_ll_bits(
    states,
    mean_bits,
    sem_bits,
    title="",
    best_k=None,
    train=None,
    ax=None,
):
    if ax is None:
        ax = plt.gca()

    TEST_C, TRAIN_C = "tab:blue", "tab:orange"
    test_line = ax.errorbar(states, mean_bits, yerr=sem_bits, marker="o", markersize=9, linewidth=2.5, capsize=4, color=TEST_C, label="test")

    if best_k is not None:
        # plot a box around the best K value
        best_idx = np.where(states == best_k)[0][0]
        ax.plot(states[best_idx], mean_bits[best_idx], marker="o", markersize=15, markerfacecolor="none", markeredgecolor="red", markeredgewidth=2)
    ax.axhline(0, color="black", linewidth=1, linestyle=":")  # Bernoulli (coin-flip) null
    ax.set_xlabel("Number of States", fontsize=20)
    ax.set_ylabel("Test LL (bits/trial)", fontsize=20, color=TEST_C)
    ax.tick_params(labelsize=15)
    ax.tick_params(axis="y", labelcolor=TEST_C)
    ax.spines["top"].set_visible(False)

    handles = [test_line]
    if train is not None:
        # Train bits/trial overlaid on an independent right-hand axis (typically higher: overfitting).
        train_mean, train_sem = train
        ax2 = ax.twinx()
        train_line = ax2.errorbar(states, train_mean, yerr=train_sem, marker="s", markersize=8, linewidth=2, linestyle="--", capsize=4, color=TRAIN_C, label="train")
        ax2.set_ylabel("Train LL (bits/trial)", fontsize=20, color=TRAIN_C)
        ax2.tick_params(labelsize=15, axis="y", labelcolor=TRAIN_C)
        ax2.spines["top"].set_visible(False)
        handles.append(train_line)
    else:
        ax.spines["right"].set_visible(False)

    xticks = states.astype(int)
    if xticks is not None:
        ax.set_xticks(xticks)
    ax.legend(handles=handles, labels=[h.get_label() for h in handles], fontsize=12, loc="lower right", frameon=False)
    ax.set_title(title, fontsize=20)

In [ ]:
desired_order = ["asmHC", "Tremor_OFF", "Brady_OFF", "Tremor_ON", "Brady_ON"]
order_map = {name: i for i, name in enumerate(desired_order)}

for model_name, model_path in MODEL_PATHS.items():

    cv_type = "pooled_cv" if model_name.endswith("__global_pooled_cv") else "session_cv"

    if cv_type == "pooled_cv":
        test_ll_fn, train_ll_fn = pooled_bits, pooled_train_bits
    elif cv_type == "session_cv":
        test_ll_fn, train_ll_fn = session_bits, session_train_bits

    # Only the base <group>.pkl CV bundles; skip the <group>_final.pkl finetuned models from 5.30.
    base_pkls = [p for p in model_path.glob("*.pkl") if not p.stem.endswith("_final")]

    fig, axs = plt.subplots(1, 5, figsize=(60, 6))
    groups = {}
    for i, subtype_model in enumerate(sorted(base_pkls, key=lambda p: order_map.get(p.stem, float("inf")))):
        group_name = subtype_model.stem
        if subtype_model.stat().st_size == 0:
            print(f"  ! skipping empty/corrupt pickle (needs refitting): {model_name}/{subtype_model.name}")
            continue
        with open(subtype_model, "rb") as f:
            groups[group_name] = pickle.load(f)

        states, mean_bits, sem_bits = test_ll_fn(groups[group_name])
        _, train_mean, train_sem = train_ll_fn(groups[group_name])
        best_k, _ = parsimonious_k(states, mean_bits, sem_bits)

        groups[group_name]["best_k"] = best_k

        plot_test_ll_bits(states, mean_bits, sem_bits, title=f"{group_name}", ax=axs[i], best_k=best_k, train=(train_mean, train_sem))

    fig.suptitle(f"{model_name}", fontsize=24, y=1.1)
    fig.savefig(model_path / "model_selection.png", bbox_inches="tight")